<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Temporl Genomics Workshop**


 Ancient DNA (aDNA) analysis, presents unique challenges due to degradation, short fragment lengths, and characteristic damage patterns. This workflow let you explore a complete pipeline for processing aDNA sequencing data, from raw reads to population-level study. It integrates multiple bioinformatics tools to ensure accurate alignment, variant calling, and downstream analysis while accounting for the limitations of low-coverage data.

## Workflow Overview
* Perform quality assessment of raw reads using FastQC and summarize with MultiQC

* Trim adapters and low-quality bases using AdapterRemoval

* Map reads to a reference genome using BWA and process alignments with Samtools

* Assess DNA damage patterns and alignment quality using DamageProfiler and Qualimap

* Call SNPs using ANGSD and perform population analysis with PLINK and PCA visualization in R


In bioinformatics workflows, multiple tools are used, each with specific dependencies and version requirements. Managing these manually can lead to compatibility issues. **Conda** is an open-source package and environment management system designed to simplify installing software, managing dependencies, and creating isolated environments for projects. Hence, the first step would be to install miniconda and create a working environment where the required tools can be installed.

In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env bwa samtools fastqc adapterremoval multiqc damageprofiler angsd plink qualimap

"mkdir" command is used for creating a directory. We will be creating separate directories throughout the pipeline to keep the wokflow organised.

"cd" is for going to the required directory

"pwd" is to print the path of working directory

In [ ]:
%%bash
mkdir -p /content/workshop_data/{fastq_files,reference,trimmed_reads,damage_profiler}
cd /content/workshop_data
pwd

This step downloads the raw sequencing data (FASTQ files) for three samples Hoggar1, Niger, and Senegal-P into the working directory. The wget -c command retrieves each file from the online repository. These FASTQ files contain the raw DNA sequences along with their quality scores, which serve as the starting point for all downstream analyses. Finally, ls -lh is used to list and verify that all files have been successfully downloaded and to inspect their sizes.

In [ ]:
%%bash
cd /content/workshop_data/fastq_files


# Download
wget -c https://zenodo.org/records/19913591/files/Hoggar1_1.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Hoggar1_2.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Niger_1.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Niger_2.100K.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Senegal-P_1.100k.fastq.gz
wget -c https://zenodo.org/records/19913591/files/Senegal-P_2.100K.fastq.gz

ls -lh

This step activates the Conda environment (`ngs_env`) to ensure that all required tools, including FastQC, are available.

A directory is then created to store the quality control results, helping keep the workflow organized.

The script navigates to the folder containing the raw FASTQ files and runs FastQC on all compressed sequencing files. This generates individual reports assessing read quality, GC content, and potential issues such as adapter contamination before further processing.

This is a very important pre processing step to understand the quality of the raw reads before downstraem analysis.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

mkdir -p /content/workshop_data/fastqc_results

cd /content/workshop_data/fastq_files

fastqc *.fastq.gz -o /content/workshop_data/fastqc_results

Once we have acquired the fastqc reports, we are going to run another tool called MultiQC, which will combine all individual FastQC reports into a single, comprehensive summary. This makes it easier to compare quality metrics across all samples in one place.

 Finally, `ls *.html` confirms that the MultiQC report has been successfully generated.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/fastqc_results
multiqc .
ls *.html

In [ ]:
from IPython.display import display, HTML

with open("/content/workshop_data/fastqc_results/multiqc_report.html", "r") as f:
    display(HTML(f.read()))

After ananlysing the Multiqc reports, now we are aware of the poor quality bases and adapter contamination in our raw data. The next step would be eliminating them before further analysis, for that purpose we are going to use AdapterRemoval tool.

 The script iterates over each sample name (Hoggar1, Niger, Senegal-P) and processes paired-end FASTQ files (`_1` and `_2`) together. The `--basename` option defines the output location and prefix, while `--gzip` compresses results and `--threads 2` speeds up execution using two CPU cores.


 `--trimns` and `--trimqualities` remove ambiguous bases and low-quality ends, while `--minquality 20` ensures only reliable bases are retained and `--minlength 30` filters out very short fragments.

  The adapter sequences (`--adapter1` and `--adapter2`) are explicitly provided so the tool can accurately remove sequencing adapters that are common in short aDNA reads.
  
  The `--minadapteroverlap 1` makes adapter detection sensitive, which is important for degraded fragments.

A key feature here is `--collapse`, which merges overlapping paired-end reads into a single sequence—this is especially important for aDNA because fragments are often shorter than the read length and overlap significantly.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/fastq_files

for sample in Hoggar1 Niger Senegal-P
do
  echo "Processing $sample..."

  AdapterRemoval \
  --file1 ${sample}_1.100*.fastq.gz \
  --file2 ${sample}_2.100*.fastq.gz \
  --basename /content/workshop_data/trimmed_reads/${sample} \
  --gzip \
  --threads 2 \
  --qualitymax 41 \
  --collapse \
  --trimns \
  --trimqualities \
  --adapter1 AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC \
  --adapter2 AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGTA \
  --minlength 30 \
  --minquality 20 \
  --minadapteroverlap 1

done

This step combines all processed read outputs from AdapterRemoval into a single file for each sample. Different categories of reads (collapsed, truncated, singleton, and paired reads) are concatenated using the cat command to ensure no usable data is lost. This creates one unified FASTQ file per sample, simplifying downstream alignment. Finally, ls -lh verifies that the combined files were successfully created and shows their sizes.

In [ ]:
%%bash
cd /content/workshop_data/trimmed_reads

for sample in Hoggar1 Niger Senegal-P
do
  echo "Combining files for $sample..."

  cat ${sample}.collapsed.gz \
      ${sample}.collapsed.truncated.gz \
      ${sample}.singleton.truncated.gz \
      ${sample}.pair1.truncated.gz \
      ${sample}.pair2.truncated.gz \
      > ${sample}.trimmed_combined.fastq.gz

done
ls -lh *.trimmed_combined.fastq.gz

This step downloads the reference genome and its index files, which are essential for aligning sequencing reads. During mapping, tools like BWA compare each read to the reference genome to determine its most likely genomic position. Without a reference, reads cannot be placed, and downstream analyses like variant calling would not be possible. Index files are precomputed data structures that allow fast and efficient searching of the reference during alignment.

###  Main Reference File

* **`chr31.fasta`**: The actual DNA sequence of the reference chromosome. All reads are aligned against this sequence.

### Index Files (used by BWA and other tools)

* **`.amb`**: Marks ambiguous bases (e.g., N’s) in the reference.
* **`.ann`**: Stores annotation about sequence names and lengths.
* **`.bwt`**: Core Burrows-Wheeler Transform index used for fast alignment.
* **`.pac`**: Packed version of the reference sequence (compressed format).
* **`.sa`**: Suffix array index that helps locate matches quickly.
* **`.fai`**: FASTA index used for quick random access to specific regions (used by tools like Samtools).
* **`.dict`**: Sequence dictionary file required by some tools (e.g., GATK) for reference metadata.

In summary, the reference genome provides the template for alignment, while the index files make the mapping process fast and computationally efficient.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/reference

wget https://zenodo.org/records/19947652/files/chr31.fasta
wget https://zenodo.org/records/19947652/files/chr31.fasta.amb
wget https://zenodo.org/records/19947652/files/chr31.fasta.ann
wget https://zenodo.org/records/19947652/files/chr31.fasta.bwt
wget https://zenodo.org/records/19947652/files/chr31.fasta.dict
wget https://zenodo.org/records/19947652/files/chr31.fasta.fai
wget https://zenodo.org/records/19947652/files/chr31.fasta.pac
wget https://zenodo.org/records/19947652/files/chr31.fasta.sa

In [ ]:
%%bash
mkdir -p /content/workshop_data/mapped_reads

This step performs read alignment using BWA, specifically the `bwa aln` algorithm. The loop processes each sample individually, taking the cleaned and combined FASTQ file and aligning it to the reference genome (`chr31.fasta`).

 The output is a `.sai` file (suffix array index), which stores intermediate alignment information and will be used in the next step to generate the final SAM file.

 `-t 2` uses two threads for faster computation, while `-n 0.01` allows a higher mismatch rate to account for sequencing errors and post-mortem damage.

 The `-l 1024` option disables seeding, which improves alignment sensitivity for short and fragmented reads, and `-o 2` allows small gaps (insertions/deletions) during alignment.

Overall, this step maps sequencing reads to their most likely positions in the reference genome, producing alignment coordinates that are essential for downstream processing such as variant calling and quality assessment.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Running bwa aln for $sample..."

  bwa aln \
  -t 2 \
  -n 0.01 \
  -l 1024 \
  -o 2 \
  reference/chr31.fasta \
  trimmed_reads/${sample}.trimmed_combined.fastq.gz \
  > mapped_reads/${sample}.sai

done

This step uses BWA (bwa samse) to convert the intermediate .sai files into SAM format, which contains the final alignment information. It takes the reference genome, the .sai alignment data, and the original trimmed reads as input. The output .sam file stores detailed information about where each read maps on the reference genome. This format is human-readable and serves as the basis for further processing like sorting and conversion to BAM.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Generating SAM for $sample..."

  bwa samse \
  reference/chr31.fasta \
  mapped_reads/${sample}.sai \
  trimmed_reads/${sample}.trimmed_combined.fastq.gz \
  > mapped_reads/${sample}.sam

done

In [ ]:
%%bash
mkdir -p /content/workshop_data/aligned_bam


A loop iterates over three samples (Hoggar1, Niger, Senegal-P) and processes each one.
For each sample, Samtools sort converts .sam files into sorted .bam files and saves them in the aligned_bam folder.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Sorting $sample..."

  samtools sort \
  -O bam \
  mapped_reads/${sample}.sam \
  -o aligned_bam/${sample}.bam

done

For each BAM file, Samtools index creates a .bai index file to enable fast data access.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing $sample..."

  samtools index aligned_bam/${sample}.bam

done

This step filters aligned reads using Samtools to retain only high-quality and reliable mappings. The `-F 4 `option removes unmapped reads, while `-q 30` keeps reads with high mapping quality, reducing false alignments. The `-h ` flag preserves the header, and `-b` outputs the result in compressed BAM format. This filtering is important to ensure that downstream analyses, such as variant calling, are based only on confident and biologically meaningful data.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Filtering $sample..."

  samtools view \
  -h \
  -F 4 \
  -q 30 \
  -b \
  aligned_bam/${sample}.bam \
  > aligned_bam/${sample}.filtered.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing filtered $sample..."

  samtools index aligned_bam/${sample}.filtered.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Removing duplicates for $sample..."

  samtools markdup \
  -r \
  -s \
  aligned_bam/${sample}.filtered.bam \
  aligned_bam/${sample}.filtered.dedup.bam

done

This step removes duplicate reads using Samtools (markdup), which is important for reducing biases introduced during PCR amplification. If not removed, these duplicates can inflate coverage and lead to incorrect variant calling.

In the script, a loop processes each sample and applies samtools markdup to the filtered BAM file. The `-r` flag removes duplicate reads rather than just marking them, ensuring they are excluded from downstream analysis. The `-s` option provides statistics about duplicate removal, which helps assess the extent of duplication in the dataset.

This step is crucial for ensuring that each DNA fragment is represented only once, improving the accuracy of SNP calling and downstream population analyses.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing deduplicated $sample..."

  samtools index aligned_bam/${sample}.filtered.dedup.bam

done

This step analyzes post-mortem DNA damage patterns using DamageProfiler for each sample. The script loops through all samples and runs the tool on the filtered, deduplicated BAM files, using the reference genome for comparison.

The `-i` parameter specifies the input BAM file, while `-r` provides the reference genome needed to assess mismatches. The `-t 2` option allows multi-threading for faster processing, and `-o` defines the output directory where damage reports (HTML and plots) are generated. The `-yaxis_dp_max 0.30` sets the maximum y-axis limit for damage plots, making patterns easier to visualize.

This step is critical in ancient DNA analysis because it identifies characteristic damage signatures such as C→T substitutions at the 5′ ends and G→A at the 3′ ends of reads. These patterns help confirm the authenticity of aDNA and distinguish it from modern contamination. Overall, DamageProfiler provides both quantitative and visual evidence of DNA degradation, which is essential for validating downstream analyses.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Running DamageProfiler for $sample..."

  conda run -n ngs_env damageprofiler \
  -i aligned_bam/${sample}.filtered.dedup.bam \
  -r reference/chr31.fasta \
  -t 2 \
  -o damage_profiler/${sample} \
  -yaxis_dp_max 0.30

done

This step evaluates the quality of the final aligned BAM files using Qualimap (`bamqc`).

The input to Qualimap is the filtered and deduplicated BAM file.

The `-bam` option specifies the input file, `-nt 2` enables multithreading for faster execution, and `-outdir` defines a separate output folder for each sample to keep results organized. The `-outformat HTML` generates an interactive report that can be easily viewed in a browser.

Qualimap provides detailed metrics such as coverage distribution, mapping quality, GC bias, and read distribution across the genome. This step is important because it gives a comprehensive overview of alignment quality after all preprocessing steps, helping to identify any remaining biases or issues before proceeding to variant calling and downstream analyses.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

mkdir -p /content/workshop_data/qualimap_results

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Running Qualimap BAMQC for $sample..."

  qualimap bamqc \
    -bam aligned_bam/${sample}.filtered.dedup.bam \
    -nt 2 \
    -outdir qualimap_results/${sample} \
    -outformat HTML

done

This step prepares the input required for downstream analysis with ANGSD by creating a list of BAM files. The command ls `aligned_bam/*.filtered.dedup.bam > bam.list `collects the paths of all processed BAM files and saves them into a single text file. Tools like ANGSD do not take multiple BAM files directly; instead, they require a file containing a list of input files.

The `cat bam.list` command is used to verify that all expected samples are correctly included in the list. This step is important because it ensures that all samples are analyzed together, enabling comparative analyses such as SNP calling and population genetics. Without this file, ANGSD would not be able to process multiple samples in a single run.

In [ ]:
%%bash
cd /content/workshop_data

ls aligned_bam/*.filtered.dedup.bam > bam.list

cat bam.list

**ANGSD:**
 (Analysis of Next Generation Sequencing Data) is a software designed for analyzing low-coverage sequencing data without requiring explicit genotype calls. It uses probabilistic models to estimate genotype likelihoods directly from sequencing reads.

**Why SNP calling is done & its importance :**

SNP calling identifies single nucleotide variations between samples, which are the basis of genetic diversity analysis. These variants allow us to compare individuals and study evolutionary relationships and population structure.

This step performs SNP calling using ANGSD, which is specifically designed for low-coverage sequencing data such as ancient DNA. The tool takes a list of BAM files (`bam.list`) and the reference genome to infer genetic variants across all samples simultaneously. The output is stored in the `angsd_results` directory with the prefix `Pigeons_snp`, generating files such as `.haplo.gz`, `.log`, and `.arg`.

The parameter `-doHaploCall 1` enables pseudo-haploid SNP calling, where one allele is randomly sampled per site—this is crucial for aDNA because coverage is often too low for reliable diploid genotype calling. The `-GL 2` option uses a GATK-like model to estimate genotype likelihoods, accounting for sequencing errors. Additional filters like `-minMapQ 30` and `-minQ 20` ensure that only high-quality reads and bases are considered, while `-uniqueOnly 1` and `-remove_bads 1` remove ambiguous and low-quality alignments.

The `-doCounts 1` option calculates read depth and allele counts, and `-doMajorMinor 1` infers the major and minor alleles at each site. The `-skipTriallelic 1` ensures only biallelic SNPs are retained, which simplifies downstream analysis.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

mkdir -p angsd_results

angsd \
  -bam bam.list \
  -out angsd_results/Pigeons_snp \
  -ref reference/chr31.fasta \
  -doHaploCall 1 \
  -doCounts 1 \
  -nThreads 2 \
  -minMapQ 30 \
  -minQ 20 \
  -uniqueOnly 1 \
  -remove_bads 1 \
  -skipTriallelic 1 \
  -doMajorMinor 1 \
  -GL 2

This step converts the SNP output from ANGSD into a format compatible with PLINK using the haploToPlink utility. The input file Pigeons_snp.haplo.gz contains pseudo-haploid SNP data generated from low-coverage sequencing. The command transforms this into PLINK text files (.tped and .tfam), which are required for downstream population genetic analyses. This conversion is important because tools like PLINK cannot directly read ANGSD output formats

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

haploToPlink Pigeons_snp.haplo.gz Pigeons_snp

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --tfile Pigeons_snp \
  --make-bed \
  --out Pigeons_snp \
  --allow-extra-chr

This step performs SNP filtering using PLINK to improve the quality of the dataset before downstream analysis. The `--bfile Pigeons_snp` option loads the binary PLINK files generated earlier, and `--maf 0.01` removes variants with a minor allele frequency less than 1%, which are often uninformative or prone to error. The `--make-bed` command creates a new set of filtered binary files, saved with the prefix `Pigeons_snp_mf01`. The `--allow-extra-chr` option ensures compatibility with non-standard chromosome names.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --bfile Pigeons_snp \
  --maf 0.01 \
  --make-bed \
  --out Pigeons_snp_mf01 \
  --allow-extra-chr

This step performs linkage disequilibrium (LD) pruning using PLINK to remove correlated SNPs from the dataset. The `--indep-pairwise 50 10 0.4 `option scans SNPs in windows of 50 variants, shifts the window by 10 SNPs, and removes one SNP from pairs with high correlation (r² > 0.4).

This is important because many nearby SNPs are not independent due to linkage, and including them can bias downstream analyses like PCA. By retaining only relatively independent variants, LD pruning reduces redundancy and noise in the dataset.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --bfile Pigeons_snp_mf01 \
  --indep-pairwise 50 10 0.4 \
  --out LD_prune \
  --allow-extra-chr

This step uses PLINK to create a new dataset containing only the SNPs that passed linkage disequilibrium (LD) pruning. The `--extract LD_prune.prune.in` option selects the list of independent SNPs identified in the previous step. The `--make-bed` command generates a new set of binary PLINK files with these filtered variants, saved as Pigeons_snp_mf01_LDpruned.

This step is important because it ensures that only uncorrelated, high-quality SNPs are used for downstream analyses like PCA, improving accuracy and reducing bias.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --bfile Pigeons_snp_mf01 \
  --extract LD_prune.prune.in \
  --make-bed \
  --out Pigeons_snp_mf01_LDpruned \
  --allow-extra-chr

This step performs Principal Component Analysis (PCA) using PLINK on the LD-pruned SNP dataset.

 The `--bfile Pigeons_snp_mf01_LDpruned` option inputs the filtered, independent variants, and `--pca 10 `computes the top 10 principal components that capture genetic variation across samples. The results are saved with the prefix pca, producing files like .eigenvec (sample coordinates) and .eigenval (variance explained). This step produces results from  PCA  which reduces high-dimensional SNP data into a few components, allowing visualization and interpretation of population structure and genetic relationships.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/angsd_results

plink \
  --bfile Pigeons_snp_mf01_LDpruned \
  --pca 10 \
  --out pca \
  --allow-extra-chr

This step creates an R script (`pca_plot.R`) that visualizes the PCA results generated by PLINK. The script first loads the PCA output files (`pca.eigenvec` and `pca.eigenval`), assigns column names, and calculates the percentage of variance explained by each principal component. It then defines population labels for the samples and combines all information into a single data frame for plotting.

The script uses R packages ggplot2 and ggrepel to generate a scatter plot of PC1 vs PC2. Points are colored by population, and sample names are added as labels without overlapping. The axes are annotated with the percentage of variance explained, making the plot more informative.

Finally, the plot is saved as a PNG file (`pca_plot.png`) for easy sharing and also printed to the screen for immediate visualization.


In [ ]:
%%bash
cat << 'EOF' > /content/workshop_data/angsd_results/pca_plot.R
# Load PCA files
pca <- read.table("pca.eigenvec", header = FALSE)
colnames(pca) <- c("FID", "IID", "PC1", "PC2")

eig <- scan("pca.eigenval")
var_exp <- eig / sum(eig) * 100

# Adjust population labels (based on your 3 samples)
pop <- factor(c("Sahara", "West Africa", "West Africa"))

# Combine data
df <- cbind(pca, pop = pop, label = pca$IID)

# Load libraries
library(ggplot2)
library(ggrepel)

# Plot PCA
p <- ggplot(df, aes(PC1, PC2, colour = pop)) +
  geom_point(size = 3.5) +
  geom_text_repel(aes(label = label)) +
  theme_minimal() +
  labs(color = "") +
  xlab(paste0("PC1 (", round(var_exp[1], 2), "%)")) +
  ylab(paste0("PC2 (", round(var_exp[2], 2), "%)"))

# Save plot
ggsave("pca_plot.png", plot = p, width = 6, height = 5)

# Also print to screen
print(p)
EOF

In [ ]:
%%bash
Rscript -e 'install.packages(c("ggplot2","ggrepel"), repos="https://cloud.r-project.org")'

In [ ]:
%%bash
cd /content/workshop_data/angsd_results
Rscript pca_plot.R

In [ ]:
from IPython.display import Image
Image("/content/workshop_data/angsd_results/pca_plot.png")